### PURPOSE AND DETAILS
- We're testing whether SEC XBRL facts can check a specified financial claim made by FinReflectKG's knowlege graph
- Scope: Apple's R&D expenses, fiscal 2024 vs 2023
- Only a manual example, not an automated verifier of multiple KG financial claims

In [75]:
from edgar import Company, set_identity

set_identity("Kedar Haldar kedar.haldar11@gmail.com")

Search the downloaded Parquet files for the first file containing ticker AAPL. Counts below describe Apple records in that file, not the complete dataset. The selected filename is recorded for reproduction.


In [76]:
from pathlib import Path
import pandas as pd

TAG = "AAPL"

root = Path(r"C:\Users\kedar\OneDrive\Documents\GitHub\Finance-GraphRag")
folder = root / "data" / "finreflectkg" / "data"

files = sorted(folder.glob("*.parquet"))

# print(len(files))       # Number of files
# print(files[0].name)    # First filename
print(folder.exists())


for file in files:
    df = pd.read_parquet(file)
    apple_rows = df[df["ticker"].astype(str).str.upper() == TAG]

    if len(apple_rows) > 0:
        selected_parquet = file
        print("Selected Parquet:", selected_parquet.name)
        print(f"{TAG} triples in this file:", len(apple_rows))
        display(apple_rows["relationship"].value_counts().head(15))
        break

True
Selected Parquet: train-00001-of-00103.parquet
AAPL triples in this file: 21036


relationship
discloses             8432
negatively_impacts    1618
depends_on            1507
subject_to             794
introduces             783
impacted_by            657
increase               622
operates_in            515
produce                509
face                   491
announces              319
decrease               317
positively_impacts     309
has_stake_in           275
invests_in             275
Name: count, dtype: int64

Locating a specific triple with which to do the test. We select a triple that claims Apple's R&D expenses rose between 2023 and 2024.

In [77]:
apple_changes = apple_rows[apple_rows["relationship"].isin(["increase", "decrease"])]

print("Matching triples:", len(apple_changes))

# display(apple_changes[[
#     "triplet_id", "entity", "relationship", "target",
#     "start_date", "end_date", "source_file"
# ]].head(10))

apple_changes = apple_changes[apple_changes["year"].astype(str).isin(["2023", "2024"])]

print("Recent claims:", len(apple_changes))

display(
    apple_changes[["triplet_id", "relationship", "target", "year", "source_file"]].head(
        15
    )
)


# for _, row in changes.iterrows():
#     print("ID:", row["triplet_id"])
#     print("Claim:", row["entity"], "→", row["relationship"], "→", row["target"])
#     print("Dates:", row["start_date"], "→", row["end_date"])
#     print("Source:", row["source_file"])
#     print("Text:", row["chunk_text"])
#     print("-" * 80)

Matching triples: 939
Recent claims: 158


,triplet_id,relationship,target,year,source_file
125033,aapl_2024_page_22_chunk_3_triplet_2,increase,share repurchase authorization,2024,AAPL_10k_2024.pdf
125034,aapl_2024_page_22_chunk_3_triplet_3,increase,share repurchase program,2024,AAPL_10k_2024.pdf
125087,aapl_2024_page_18_chunk_1_triplet_10,increase,substantial cost,2024,AAPL_10k_2024.pdf
125106,aapl_2024_page_18_chunk_1_triplet_29,increase,high net sale,2024,AAPL_10k_2024.pdf
125143,aapl_2024_page_26_chunk_2_triplet_7,increase,total net sale,2024,AAPL_10k_2024.pdf
125149,aapl_2024_page_26_chunk_2_triplet_14,increase,service revenue,2024,AAPL_10k_2024.pdf
125150,aapl_2024_page_25_chunk_3_triplet_1,increase,net sale,2024,AAPL_10k_2024.pdf
125158,aapl_2024_page_25_chunk_3_triplet_9,decrease,ipad sale,2024,AAPL_10k_2024.pdf
125222,aapl_2024_page_27_chunk_5_triplet_1,increase,r & d expense,2024,AAPL_10k_2024.pdf
125223,aapl_2024_page_27_chunk_5_triplet_2,increase,headcount expense,2024,AAPL_10k_2024.pdf


In [78]:
candidate_rows = apple_rows.loc[
    apple_rows["triplet_id"] == "aapl_2024_page_27_chunk_5_triplet_1"
]
assert len(candidate_rows) == 1, "Expected one unique KG claim"
candidate = candidate_rows.iloc[0]
print(candidate[["triplet_id", "entity", "relationship", "target",
                 "start_date", "end_date", "source_file"]].to_string())
print("Source passage:", candidate["chunk_text"])


triplet_id      aapl_2024_page_27_chunk_5_triplet_1
entity                                         aapl
relationship                               increase
target                                r & d expense
start_date                             January 2024
end_date                              December 2024
source_file                       AAPL_10k_2024.pdf
Source passage: ## Research and Development The growth in R&amp;D expense during 2024 compared to 2023 was driven primarily by increases in headcount-related expenses. ## Selling, General and Administrative Selling, general and administrative expense increased $1.2 billion during 2024 compared to 2023.


Search for concept names containing Research OR Development (`|` means OR). This produces three candidate concepts. We manually select `us-gaap:ResearchAndDevelopmentExpense` because it represents the expense in the claim; the other two concern research tax credits. This is a human-reviewed mapping, not automated concept alignment.


In [79]:
apple = Company("AAPL")
apple_facts = apple.get_facts()
aapl_df = apple_facts.query().to_dataframe()

r_and_d_concepts = aapl_df[
    aapl_df["concept"].str.contains("Research|Development", case=False, na=False)
][["concept", "label"]].drop_duplicates()

display(r_and_d_concepts)
print(r_and_d_concepts.to_string(index=False))

,concept,label
5743,us-gaap:DeferredTaxAssetsTaxCreditCarryforward...,"Deferred Tax Assets, Tax Credit Carryforwards,..."
9766,us-gaap:IncomeTaxReconciliationTaxCreditsResearch,"Effective Income Tax Rate Reconciliation, Tax ..."
20414,us-gaap:ResearchAndDevelopmentExpense,Research and Development Expense


                                                concept                                                                  label
us-gaap:DeferredTaxAssetsTaxCreditCarryforwardsResearch                Deferred Tax Assets, Tax Credit Carryforwards, Research
      us-gaap:IncomeTaxReconciliationTaxCreditsResearch Effective Income Tax Rate Reconciliation, Tax Credit, Research, Amount
                  us-gaap:ResearchAndDevelopmentExpense                                       Research and Development Expense


Select full fiscal-year R&D expenses for FY2023 and FY2024 from accession `0000320193-24-000123`, filed November 1, 2024. Both facts use USD and the same filing version. Check exact start/end dates rather than assuming that two rows must be annual observations.

The comparison interprets the source passage's 2024 versus 2023 as fiscal years. The KG's stored dates are displayed separately; they are not silently replaced. Exact KG-date alignment still needs manual review. The dataset label `AAPL_10k_2024.pdf` has not been conclusively linked to this SEC accession through the original PDF or source metadata.


In [80]:
concept = "us-gaap:ResearchAndDevelopmentExpense"

matches = aapl_df[
    (aapl_df["concept"] == concept)
    & pd.to_datetime(aapl_df["period_end"]).dt.year.isin([2023, 2024])
    & (aapl_df["form_type"] == "10-K")
]

# Inspect available filing versions; pin the selected 2024 accession below

# print(
#     matches[
#         [
#             "period_start",
#             "period_end",
#             "numeric_value",
#             "unit",
#             "filing_date",
#             "accession",
#         ]
#     ]
#     .sort_values(["filing_date", "period_end"])
#     .to_string(index=False)
# )

filing_id = "0000320193-24-000123"

pair = matches[
    (matches["accession"] == filing_id)
    & (matches["unit"] == "USD")
].sort_values("period_end")

pair = pair.copy()
pair["period_start"] = pd.to_datetime(pair["period_start"])
pair["period_end"] = pd.to_datetime(pair["period_end"])
expected_periods = [
    (pd.Timestamp("2022-09-25"), pd.Timestamp("2023-09-30")),
    (pd.Timestamp("2023-10-01"), pd.Timestamp("2024-09-28")),
]
assert list(zip(pair["period_start"], pair["period_end"])) == expected_periods, \
    "Expected exactly the two selected full fiscal-year periods"
assert pair["numeric_value"].notna().all(), "Missing expense value"

print(pair[[
    "period_start", "period_end", "numeric_value", "unit", "accession"
]].to_string(index=False))





period_start period_end  numeric_value unit            accession
  2022-09-25 2023-09-30   2.991500e+10  USD 0000320193-24-000123
  2023-10-01 2024-09-28   3.137000e+10  USD 0000320193-24-000123


The saved comparison found a positive difference of **$1.455 billion ($1,455,000,000)**: $31.370 billion for FY2024 minus $29.915 billion for FY2023. This supports the KG's increase relationship for the selected fiscal periods, conditional on the manual mapping and source alignment. Rerunning the next cell recalculates the verdict.

The source passage's **$1.2 billion** amount refers to **SG&A**, not R&D. The selected R&D triple claims a direction, not an amount. This check does not establish that headcount-related expenses caused the increase.


In [81]:
difference = pair["numeric_value"].iloc[1] - pair["numeric_value"].iloc[0]

print(f"Changes in R&D spending: ${difference:,.0f}")

if candidate["relationship"] == "increase":
    print("Supports 'increase':", difference > 0)
elif candidate["relationship"] == "decrease":
    print("Supports 'decrease': ", difference < 0)
else:
    print("Relationship not handled by this check.")

Changes in R&D spending: $1,455,000,000
Supports 'increase': True


### Limits and Findings
- The XBRL check supports the claim that Apple's R&D expenses increased, but does not provide the explanation that headcount caused it. (this explanation comes from the 10K source)
- Concept and period selection required human judgment.
- One supported claim does not establish verification accuracy or coverage across the graph.
- KG date fields must be reviewed against the fiscal periods printed above; fiscal-year direction support does not validate those fields.
- Exact identity between the dataset's source PDF and the selected SEC filing is not yet confirmed.
- This uses SEC Company Facts derived from XBRL, not a full filing-level XBRL context or calculation-linkbase validation.
- The next cell saves the selected inputs, result, runtime versions, and execution time. It does not freeze the entire live SEC response or the full Parquet dataset.


In [82]:
from datetime import datetime
from zoneinfo import ZoneInfo
from importlib.metadata import version
import platform
import json
import hashlib

run_time = datetime.now(ZoneInfo("America/Los_Angeles"))
output_dir = root / "data" / "mini_xbrl_results" / run_time.strftime("%Y%m%dT%H%M%S%f")
output_dir.mkdir(parents=True, exist_ok=False)
pair.to_csv(output_dir / "selected_xbrl_facts.csv", index=False)
candidate.to_json(output_dir / "selected_kg_claim.json", indent=2)
relationship = candidate["relationship"]
supported = bool(difference > 0) if relationship == "increase" else (
    bool(difference < 0) if relationship == "decrease" else None
)
record = {
    "run_time": run_time.isoformat(),
    "python": platform.python_version(),
    "packages": {name: version(name) for name in ["pandas", "edgartools", "pyarrow"]},
    "parquet_file": str(selected_parquet),
    "triplet_id": candidate["triplet_id"],
    "concept": concept,
    "accession": filing_id,
    "difference_usd": float(difference),
    "direction_supported": supported,
    "kg_date_alignment": "Pending manual review; see saved KG dates and XBRL periods",
    "exact_source_filing_identity": "Not independently confirmed",
    "saved_input_sha256": {
        name: hashlib.sha256((output_dir / name).read_bytes()).hexdigest()
        for name in ["selected_xbrl_facts.csv", "selected_kg_claim.json"]
    },
}
(output_dir / "run_record.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
print("Saved test inputs and result to:", output_dir)


Saved test inputs and result to: C:\Users\kedar\OneDrive\Documents\GitHub\Finance-GraphRag\data\mini_xbrl_results\20261022T154841215318
